### cleaning ideology data so that it can be added to the senator data

In [25]:
# imports
import pandas as pd

In [26]:
# load ideology score files
nominate = pd.read_csv('../data/ideology_scores/nominate.csv')
govtrack  = pd.read_csv('../data/ideology_scores/govtrack.csv')

### nominate

congress — which congress number they served in (1 = 1789, 119 = current)

chamber — Senate, House, or President

icpsr — a numeric ID from the Inter-university Consortium for Political and Social Research, an alternative to bioguide_id

state_icpsr — numeric code for the state (not the abbreviation)

district_code — House district number (0 for Senate)

state_abbrev — two-letter state abbreviation

party_code — numeric party code (100 = Democrat, 200 = Republican, 5000 = various independent/third party)

occupancy — whether they were the 1st or 2nd senator from their state in that congress

last_means — how their service ended (resigned, died, lost reelection, etc.)

bioname — name in LAST, First Middle format

bioguide_id — the bioguide ID that matches your senators df

born / died — birth and death years

nominate_dim1 — the main ideology score: negative = liberal, positive = conservative

nominate_dim2 — secondary dimension, historically captured cross-cutting issues like race/civil rights

nominate_log_likelihood — model fit stat, how well their votes fit the estimated position

nominate_geo_mean_probability — another model fit stat, higher = votes were more predictable

nominate_number_of_votes — how many votes they cast that congress

nominate_number_of_errors — how many votes didn't match their predicted position

conditional — flag for a specific estimation condition, usually ignorable

nokken_poole_dim1/2 — an alternative ideology score that allows positions to shift congress-to-congress (vs NOMINATE which fixes one score per person)

In [27]:
# drop rows with no bioguide_id or no scores
nominate = nominate.dropna(subset=['bioguide_id', 'nominate_dim1'])

# keep only senate rows (you have house + president in there too)
nominate = nominate[nominate['chamber'] == 'Senate']

# one row per person — average their scores across all congresses they served
nominate_clean = nominate.groupby('bioguide_id').agg(
    nominate_dim1=('nominate_dim1', 'mean'),
    nominate_dim2=('nominate_dim2', 'mean')
).reset_index()

# parse the name into first and last name
def parse_bioname(name):
    if not isinstance(name, str):
        return None, None
    # format is "LAST, First Middle" — split on first comma
    parts = name.split(',', 1)
    last = parts[0].strip().title()  # "ELLSWORTH" → "Ellsworth"
    first = parts[1].strip().title() if len(parts) > 1 else None  # "Oliver" 
    return first, last

nominate[['first_name', 'last_name']] = nominate['bioname'].apply(
    lambda x: pd.Series(parse_bioname(x))
)

# drop the original bioname col if you don't need it anymore
nominate = nominate.drop(columns=['bioname'])

In [28]:
nominate.head()

,congress,chamber,icpsr,state_icpsr,district_code,state_abbrev,party_code,occupancy,last_means,bioguide_id,...,nominate_dim2,nominate_log_likelihood,nominate_geo_mean_probability,nominate_number_of_votes,nominate_number_of_errors,conditional,nokken_poole_dim1,nokken_poole_dim2,first_name,last_name
67,1,Senate,2936,1,0.0,CT,5000,0.0,3.0,E000147,...,0.809,-24.37915,0.778,97.0,8.0,NaN,0.528,0.849,Oliver,Ellsworth
68,1,Senate,4998,1,0.0,CT,5000,0.0,3.0,J000182,...,0.137,-30.41227,0.690,82.0,16.0,NaN,0.997,0.075,William Samuel,Johnson
69,1,Senate,507,11,0.0,DE,4000,0.0,3.0,B000226,...,0.007,-38.18355,0.654,90.0,23.0,NaN,0.024,0.166,Richard,Bassett
70,1,Senate,7762,11,0.0,DE,5000,0.0,3.0,R000091,...,-0.239,-34.31907,0.699,96.0,15.0,NaN,0.270,-0.206,George,Read
71,1,Senate,3128,44,0.0,GA,4000,0.0,3.0,F000100,...,-0.433,-39.85431,0.626,85.0,27.0,NaN,-0.411,-0.515,William,Few


### govtrack

rank_from_low — rank ordered from most liberal (1) to most conservative. Sanders is 1 = most liberal

rank_from_high — rank ordered from most conservative (1) to most liberal. Flip side of rank_from_low

percentile — what percentile they fall in from the liberal end. Sanders = 0th percentile (most liberal), so a high percentile = more conservative

ideology — the actual GovTrack ideology score, ranges from 0.0 (most liberal) to 1.0 (most conservative). This is the key column you want

id — GovTrack's internal numeric ID, not useful for merging with your data

bioguide_id — the bioguide ID, matches your senators df — this is your merge key

state — two-letter state abbreviation

district — House district number, blank for senators

name — last name only, no first name

In [29]:
# clean the name col — strip the b'' bytes literal formatting
govtrack['name'] = govtrack['name'].str.replace(r"^b'|'$", "", regex=True)

# only need these cols
govtrack_clean = govtrack[['bioguide_id', 'ideology']].rename(
    columns={'ideology': 'govtrack_ideology_score'}
)

In [33]:
# get first/last name from nominate (already has bioguide_id + parsed names)
name_lookup = nominate[['bioguide_id', 'first_name', 'last_name']].drop_duplicates(subset='bioguide_id')

# add names to govtrack
govtrack = govtrack.merge(name_lookup, on='bioguide_id', how='left')

In [34]:
govtrack.head()

,rank_from_low,rank_from_high,percentile,ideology,id,bioguide_id,state,district,name,first_name_x,last_name_x,first_name_y,last_name_y
0,1,100,0,0.000000,400357,S000033,VT,NaN,Sanders,Bernard,Sanders,Bernard,Sanders
1,2,99,1,0.056155,412239,W000800,VT,NaN,Welch,Peter,Welch,Peter,Welch
2,3,98,2,0.072088,412325,M001176,OR,NaN,Merkley,Jeff,Merkley,Jeff,Merkley
3,4,97,3,0.083912,412490,B001277,CT,NaN,Blumenthal,Richard,Blumenthal,Richard,Blumenthal
4,5,96,4,0.083938,412542,W000817,MA,NaN,Warren,Elizabeth,Warren,Elizabeth,Warren


### saving cleaned dfs as new CSVs

In [35]:
nominate.to_csv('../data/ideology_scores/nominate_cleaned.csv', index=False)
govtrack.to_csv('../data/ideology_scores/govtrack_cleaned.csv', index=False)